<a href="https://colab.research.google.com/github/OdysseusPolymetis/enssib_class/blob/main/03_weat_genre_ethnicite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gensim pandas numpy matplotlib transformers accelerate diffusers safetensors sentencepiece

# 03 — WEAT simplifié : genre, carrière/famille, noms ethniques

Objectif : reproduire l’idée générale du Word Embedding Association Test : comparer deux groupes de mots à deux familles d’attributs.

In [ ]:
import gensim.downloader as api
import numpy as np, pandas as pd
model = api.load('word2vec-google-news-300')

def cos(a,b):
    return float(np.dot(model[a], model[b])/(np.linalg.norm(model[a])*np.linalg.norm(model[b])))

def assoc(w, A, B):
    return np.mean([cos(w,a) for a in A if a in model]) - np.mean([cos(w,b) for b in B if b in model])

def effect_size(X, Y, A, B):
    xs=[assoc(x,A,B) for x in X if x in model]
    ys=[assoc(y,A,B) for y in Y if y in model]
    return (np.mean(xs)-np.mean(ys))/np.std(xs+ys, ddof=1)


In [ ]:
male = ['he','him','man','male','boy','father','son']
female = ['she','her','woman','female','girl','mother','daughter']
career = ['career','corporation','salary','office','professional','management','business']
family = ['home','parents','children','family','cousins','marriage','wedding']
print('Effet genre / carrière vs famille :', effect_size(male, female, career, family))

Dans cet espace vectoriel GNews, les termes masculins sont en moyenne davantage associés à "carrière plutôt que famille" que les termes féminins.

In [ ]:
euro_names = ['mark','lucius','winston','albert','leonard','howard','bernadette','john','james']
african_american_names = ['darnell','hakim','jermaine','kareem','jamal','leroy','rasheed','tremayne','tyrone']
pleasant = ['clever','freedom','happyness','gentle','peace','cheer','rich','loyal']
unpleasant = ['abuse','crash','filth','murder','sickness','accident','death','grief','poison','stink', 'stupid']
print('Effet noms / pleasant vs unpleasant :', effect_size(euro_names, african_american_names, pleasant, unpleasant))

In [ ]:
rows = []

for name in euro_names:
    if name in model:
        rows.append({
            "name": name,
            "group": "European-American",
            "score": assoc(name, pleasant, unpleasant)
        })

for name in african_american_names:
    if name in model:
        rows.append({
            "name": name,
            "group": "African-American",
            "score": assoc(name, pleasant, unpleasant)
        })

scores_names = pd.DataFrame(rows)

display(
    scores_names.groupby("group", as_index=False)["score"]
    .mean()
    .rename(columns={"score": "mean_score"})
)

import matplotlib.pyplot as plt

plot_df = scores_names.sort_values("score").copy()
plot_df["label"] = plot_df["group"] + " — " + plot_df["name"]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_df["label"], plot_df["score"])
ax.axvline(0, linewidth=1)
ax.set_xlabel("Association : pleasant − unpleasant")
ax.set_ylabel("")
ax.set_title("Association positive/négative de chaque nom")
plt.tight_layout()
plt.show()